In [1]:
   import os
   print(os.getcwd())

/Users/sandeepa/delhi-aqi-forecaster/notebook


In [2]:
print(os.path.exists("../data/raw"))

True


In [3]:
import pandas as pd
import numpy as np
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.61,
    "longitude": 77.21,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
print("Status:", response.status_code)

Status: 200


In [4]:
data = response.json()
weather = pd.DataFrame(data["daily"])

weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temp_mean",
    "precipitation_sum": "rain_mm",
    "wind_speed_10m_max": "wind_speed_max",
    "wind_direction_10m_dominant": "wind_dir"
})
weather["date"] = pd.to_datetime(weather["date"])

print(weather.shape)
weather.head()

(1826, 5)


,date,temp_mean,rain_mm,wind_speed_max,wind_dir
0,2021-01-01,10.5,0.0,9.1,103
1,2021-01-02,13.7,4.5,12.9,108
2,2021-01-03,16.0,8.1,23.4,121
3,2021-01-04,16.6,5.6,14.2,96
4,2021-01-05,17.5,2.8,14.2,94


In [5]:
weather.to_csv("../data/raw/delhi_weather_raw.csv", index=False)
print("Saved!")

Saved!


In [8]:
weather.head()

,date,temp_mean,rain_mm,wind_speed_max,wind_dir
0,2021-01-01,10.5,0.0,9.1,103
1,2021-01-02,13.7,4.5,12.9,108
2,2021-01-03,16.0,8.1,23.4,121
3,2021-01-04,16.6,5.6,14.2,96
4,2021-01-05,17.5,2.8,14.2,94


In [9]:
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

aq_params = {
    "latitude": 28.61,
    "longitude": 77.21,
    "start_date": "2022-08-01",
    "end_date": "2025-12-31",
    "hourly": "pm2_5,pm10",
    "timezone": "Asia/Kolkata"
}

aq_response = requests.get(aq_url, params=aq_params)
print("Status:", aq_response.status_code)

Status: 200


In [10]:
aq = pd.DataFrame(aq_response.json()["hourly"])
aq = aq.rename(columns={"time": "datetime"})
aq["datetime"] = pd.to_datetime(aq["datetime"])

print(aq.shape)
aq.head()

(29976, 3)


,datetime,pm2_5,pm10
0,2022-08-01 00:00:00,NaN,NaN
1,2022-08-01 01:00:00,NaN,NaN
2,2022-08-01 02:00:00,NaN,NaN
3,2022-08-01 03:00:00,NaN,NaN
4,2022-08-01 04:00:00,NaN,NaN


In [11]:
aq.to_csv("../data/raw/delhi_air_quality_raw.csv", index=False)
print("Saved!")

Saved!


In [12]:
aq.head()

,datetime,pm2_5,pm10
0,2022-08-01 00:00:00,NaN,NaN
1,2022-08-01 01:00:00,NaN,NaN
2,2022-08-01 02:00:00,NaN,NaN
3,2022-08-01 03:00:00,NaN,NaN
4,2022-08-01 04:00:00,NaN,NaN


In [13]:
print("Total rows:", len(aq))
print("Missing values:")
print(aq.isna().sum())

has_data = aq[aq["pm2_5"].notna()]
print("\nRows with PM2.5:", len(has_data))
print("First date with data:", has_data["datetime"].min())
print("Last date with data:", has_data["datetime"].max())

aq.tail()

Total rows: 29976
Missing values:
datetime     0
pm2_5       77
pm10        77
dtype: int64

Rows with PM2.5: 29899
First date with data: 2022-08-04 05:00:00
Last date with data: 2025-12-31 23:00:00


,datetime,pm2_5,pm10
29971,2025-12-31 19:00:00,170.9,171.8
29972,2025-12-31 20:00:00,169.0,170.0
29973,2025-12-31 21:00:00,164.4,165.3
29974,2025-12-31 22:00:00,160.2,160.7
29975,2025-12-31 23:00:00,155.5,156.5


In [15]:
import os

print("Inside data/raw:")
print(os.listdir("../data/raw"))

print("\nAll CSV files anywhere inside data/raw:")
print(glob.glob("../data/raw/**/*.csv", recursive=True))

Inside data/raw:
['delhi_weather_raw.csv', 'modis_2023_India.csv', 'modis_2024_India.csv', 'modis_2022_India.csv', 'delhi_air_quality_raw.csv']

All CSV files anywhere inside data/raw:
['../data/raw/delhi_weather_raw.csv', '../data/raw/modis_2023_India.csv', '../data/raw/modis_2024_India.csv', '../data/raw/modis_2022_India.csv', '../data/raw/delhi_air_quality_raw.csv']


In [16]:
files = sorted(glob.glob("../data/raw/modis_*_India.csv"))

In [18]:
import glob

files = sorted(glob.glob("../data/raw/modis_*_India.csv"))
print(files)

fires = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(fires.shape)
fires.head()

['../data/raw/modis_2022_India.csv', '../data/raw/modis_2023_India.csv', '../data/raw/modis_2024_India.csv']
(233979, 15)


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,30.1138,80.0756,300.0,1.2,1.1,2022-01-01,511,Terra,MODIS,7,6.03,288.4,7.1,D,0
1,23.7726,86.2078,306.1,1.6,1.2,2022-01-01,512,Terra,MODIS,62,6.03,293.5,10.4,D,2
2,22.2080,84.8627,304.8,1.4,1.2,2022-01-01,512,Terra,MODIS,42,6.03,293.3,5.8,D,2
3,23.7621,86.3946,306.9,1.6,1.2,2022-01-01,512,Terra,MODIS,38,6.03,295.2,9.3,D,2
4,23.6787,86.0891,303.6,1.5,1.2,2022-01-01,512,Terra,MODIS,52,6.03,293.1,7.2,D,2


In [19]:
in_region = (
    fires["latitude"].between(27.5, 32.6) &
    fires["longitude"].between(73.8, 77.6)
)

fires_pb = fires[in_region]
fires_pb = fires_pb[fires_pb["type"] == 0]

fires_pb["acq_date"] = pd.to_datetime(fires_pb["acq_date"])

print("All India fires:", len(fires))
print("Punjab-Haryana fires:", len(fires_pb))
fires_pb.head()

All India fires: 233979
Punjab-Haryana fires: 40747


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
65,30.5335,75.7878,302.7,1.4,1.2,2022-01-01,831,Aqua,MODIS,53,6.03,290.3,7.6,D,0
68,30.3164,75.6704,312.2,1.4,1.2,2022-01-01,831,Aqua,MODIS,72,6.03,290.9,17.8,D,0
79,30.4390,74.3372,301.1,1.1,1.0,2022-01-02,553,Terra,MODIS,43,6.03,288.7,4.5,D,0
81,30.4337,74.3299,303.1,1.1,1.0,2022-01-02,553,Terra,MODIS,55,6.03,288.7,6.3,D,0
246,29.0031,75.9597,309.4,1.0,1.0,2022-01-03,818,Aqua,MODIS,58,6.03,292.3,8.4,D,0


In [20]:
fires_pb.to_csv("../data/processed/fires_punjab_haryana.csv", index=False)
print("Saved!")

Saved!
